# Dark-store demand forecasting workflow

This notebook follows the project workflow from chronological validation through model comparison and optional final submission creation. It uses the scripts in `script/` for the same leakage-aware feature engineering logic used elsewhere in this repository.

**Important:** run cells in order. The final full-data submission section is optional and separated from the holdout comparison. It trains a fresh model using all training data and overwrites `data/submission.csv` only when `RUN_FINAL_SUBMISSION` is set to `True`.

Expected packages in the selected environment: pandas, NumPy, LightGBM, XGBoost, and CatBoost. Generated files are written under `data/`.

In [ ]:
from pathlib import Path
import math
import sys

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import catboost
from catboost import CatBoostRegressor

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "orders_train.csv").exists():
    PROJECT_ROOT = Path.cwd().parent
if not (PROJECT_ROOT / "data" / "orders_train.csv").exists():
    raise FileNotFoundError("Run this notebook from the project root or its notebooks folder.")

DATA_DIR = PROJECT_ROOT / "data"
FORECAST_DAYS = 42
N_ESTIMATORS = 800
RANDOM_STATE = 42
print("Project:", PROJECT_ROOT)
print("pandas", pd.__version__, "NumPy", np.__version__)
print("LightGBM", lgb.__version__, "XGBoost", xgb.__version__, "CatBoost", catboost.__version__)

## 1. Load and validate the provided data

The target is `OrderVolume`. `AppSessions` is present only in historical training rows, so the feature-engineering helper removes it to avoid using a feature that is absent from future test rows.

In [ ]:
train_all = pd.read_csv(DATA_DIR / "orders_train.csv", parse_dates=["Date"])
test_raw = pd.read_csv(DATA_DIR / "orders_test.csv", parse_dates=["Date"])
metadata = pd.read_csv(DATA_DIR / "hub_metadata.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("Training:", train_all.shape, train_all.Date.min(), "to", train_all.Date.max())
print("Test:", test_raw.shape, test_raw.Date.min(), "to", test_raw.Date.max())
print("Metadata:", metadata.shape)
print("Sample submission:", sample_submission.shape, sample_submission.columns.tolist())
assert not train_all[["HubID", "Date"]].duplicated().any()
assert not test_raw[["HubID", "Date"]].duplicated().any()
assert not metadata["HubID"].duplicated().any()
assert sample_submission.columns.tolist() == ["Id", "OrderVolume"]
assert (train_all.loc[train_all.IsOpen.eq(0), "OrderVolume"] == 0).all()
display(train_all.head())
display(metadata.head())

## 2. Make the chronological 42-day holdout

The last 42 distinct dates are held out for evaluation. This is a date-based split, not a random split, so evaluation resembles the future forecast task. The original training CSV stays unchanged.

In [ ]:
all_dates = pd.DatetimeIndex(train_all.Date.drop_duplicates().sort_values())
assert len(all_dates) > FORECAST_DAYS
holdout_dates = all_dates[-FORECAST_DAYS:]
assert holdout_dates.equals(pd.date_range(holdout_dates[0], holdout_dates[-1], freq="D"))
holdout_start = holdout_dates[0]
history = train_all.loc[train_all.Date < holdout_start].copy()
evaluation = train_all.loc[train_all.Date >= holdout_start].copy()
assert history.Date.max() < evaluation.Date.min()
print(f"History: {len(history):,} rows through {history.Date.max():%Y-%m-%d}")
print(f"Evaluation: {len(evaluation):,} rows, {evaluation.Date.min():%Y-%m-%d} to {evaluation.Date.max():%Y-%m-%d}")
print("Unique evaluation dates:", evaluation.Date.nunique())
display(evaluation.groupby("Date").size().describe())

## 3. Leakage-aware feature engineering

Training examples use frozen historical snapshots in 42-day blocks. Evaluation features use only the earlier history. Historical demand averages use open days only; exact calendar lags retain closed-day zeros. Closure streak and preceding 7-/30-day closure counts use `IsOpen` and exclude the current day. Closed rows stay in evaluation for scoring, but are removed from model fitting.

In [ ]:
sys.path.insert(0, str(PROJECT_ROOT))
from script.feature_engineering import (
    build_training_features,
    build_forecast_features,
    validate_frame,
)
from script.train_lightgbm import CATEGORICAL_COLUMNS, prepare_features

validate_frame(history, "history", {"HubID", "Date", "OrderVolume", "IsOpen"})
validate_frame(evaluation, "evaluation", {"HubID", "Date", "OrderVolume", "IsOpen"})
def feature_progress(state, message, **details):
    print(message, flush=True)

train_features = build_training_features(
    history, metadata, FORECAST_DAYS, feature_progress
)
eval_features = build_forecast_features(
    history=history,
    future_rows=evaluation,
    metadata=metadata,
    first_date=history.Date.min(),
    forecast_origin=history.Date.max(),
)
closure_features = [
    "ClosedStreakBeforeDate",
    "ClosedDaysPrevious7Days",
    "ClosedDaysPrevious30Days",
]
assert all(column in train_features and column in eval_features for column in closure_features)
assert train_features.IsOpen.eq(1).all()
DATA_DIR.joinpath("orders_train_holdout_features.csv").parent.mkdir(exist_ok=True)
train_features.to_csv(DATA_DIR / "orders_train_holdout_features.csv", index=False)
eval_features.to_csv(DATA_DIR / "orders_eval_holdout_features.csv", index=False)
print("Train feature rows:", f"{len(train_features):,}")
print("Evaluation feature rows:", f"{len(eval_features):,}")
print("Closure features:", closure_features)

## 4. Prepare common model matrices and metric

All three models use identical feature columns and training/evaluation rows. Models learn `log1p(OrderVolume)`; predictions are transformed back with `expm1`, clipped at zero, and closed-hub predictions are explicitly zero. RMSLE includes every evaluation row.

In [ ]:
def rmsle(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.maximum(np.asarray(predicted, dtype=float), 0)
    if not np.isfinite(actual).all() or not np.isfinite(predicted).all():
        raise ValueError("RMSLE inputs must be finite.")
    return float(np.sqrt(np.mean((np.log1p(predicted) - np.log1p(actual)) ** 2)))

training_rows = train_features.loc[train_features.IsOpen.eq(1)].copy()
y_train = pd.to_numeric(training_rows.OrderVolume, errors="raise")
y_eval = pd.to_numeric(eval_features.OrderVolume, errors="raise")
X_train, X_eval, categorical_columns = prepare_features(training_rows, eval_features)
y_train_log = y_train.map(math.log1p)
open_eval = eval_features.IsOpen.eq(1)
print(f"Fit rows: {len(training_rows):,}; evaluation rows: {len(eval_features):,}")
print("Categorical columns:", categorical_columns)

## 5. Fit and compare LightGBM, XGBoost, and CatBoost

The initial parameters match the project scripts. This cell may take several minutes. Each model's evaluation RMSLE is calculated against the same 42-day holdout.

In [ ]:
lgb_model = lgb.LGBMRegressor(
    objective="regression", n_estimators=N_ESTIMATORS, learning_rate=0.05,
    num_leaves=31, max_depth=-1, min_child_samples=100, subsample=0.8,
    subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
    random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1,
)
lgb_model.fit(X_train, y_train_log, categorical_feature=categorical_columns)
lgb_pred = np.zeros(len(eval_features), dtype=float)
lgb_pred[open_eval.to_numpy()] = np.maximum(
    0, np.expm1(lgb_model.predict(X_eval.loc[open_eval]))
)
print("LightGBM RMSLE:", rmsle(y_eval, lgb_pred))

class XGBProgress(xgb.callback.TrainingCallback):
    def after_iteration(self, model, epoch, evals_log):
        iteration = epoch + 1
        if iteration == 1 or iteration % 50 == 0 or iteration == N_ESTIMATORS:
            print(f"XGBoost iteration {iteration}/{N_ESTIMATORS}", flush=True)
        return False

xgb_model = xgb.XGBRegressor(
    objective="reg:squarederror", n_estimators=N_ESTIMATORS,
    learning_rate=0.05, max_depth=8, min_child_weight=100,
    subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
    tree_method="hist", enable_categorical=True, n_jobs=-1,
    random_state=RANDOM_STATE, verbosity=0, callbacks=[XGBProgress()],
)
xgb_model.fit(X_train, y_train_log)
xgb_pred = np.zeros(len(eval_features), dtype=float)
xgb_pred[open_eval.to_numpy()] = np.maximum(
    0, np.expm1(xgb_model.predict(X_eval.loc[open_eval]))
)
print("XGBoost RMSLE:", rmsle(y_eval, xgb_pred))

cat_train, cat_eval = X_train.copy(), X_eval.copy()
for column in categorical_columns:
    cat_train[column] = cat_train[column].astype("string").fillna("__MISSING__")
    cat_eval[column] = cat_eval[column].astype("string").fillna("__MISSING__")
cat_model = CatBoostRegressor(
    iterations=N_ESTIMATORS, learning_rate=0.05, depth=8,
    loss_function="RMSE", random_seed=RANDOM_STATE, thread_count=-1,
    verbose=50, allow_writing_files=False,
)
cat_model.fit(cat_train, y_train_log, cat_features=categorical_columns)
cat_pred = np.zeros(len(eval_features), dtype=float)
cat_pred[open_eval.to_numpy()] = np.maximum(
    0, np.expm1(cat_model.predict(cat_eval.loc[open_eval]))
)
print("CatBoost RMSLE:", rmsle(y_eval, cat_pred))

holdout_predictions = eval_features[["HubID", "Date", "IsOpen"]].copy()
holdout_predictions["LightGBMPrediction"] = lgb_pred
holdout_predictions["XGBoostPrediction"] = xgb_pred
holdout_predictions["CatBoostPrediction"] = cat_pred
scores = pd.DataFrame({
    "Model": ["LightGBM", "XGBoost", "CatBoost"],
    "RMSLE": [rmsle(y_eval, lgb_pred), rmsle(y_eval, xgb_pred), rmsle(y_eval, cat_pred)],
}).sort_values("RMSLE")
display(scores)
holdout_predictions.to_csv(DATA_DIR / "notebook_model_comparison_predictions.csv", index=False)

## 6. Try an equal-weight ensemble

The ensemble is calculated row by row and scored using the same complete evaluation set. Compare its RMSLE to the individual models before deciding whether it is useful.

In [ ]:
ensemble_pred = (lgb_pred + xgb_pred + cat_pred) / 3
ensemble_score = rmsle(y_eval, ensemble_pred)
print(f"Equal-weight ensemble RMSLE: {ensemble_score:.8f}")
print("Best individual model:", scores.iloc[0].Model, f"({scores.iloc[0].RMSLE:.8f})")

## 7. Optional: train on all history and create final XGBoost submission

This section rebuilds forecast-style training features using the entire `orders_train.csv`, creates test features from that full history, trains XGBoost, and writes `data/submission.csv` in sample ID order. Run this only after reviewing the holdout scores. Set the switch below to `True` to execute.

In [ ]:
RUN_FINAL_SUBMISSION = False
if RUN_FINAL_SUBMISSION:
    full_dates = pd.DatetimeIndex(train_all.Date.drop_duplicates().sort_values())
    full_train_features = build_training_features(
        train_all, metadata, FORECAST_DAYS, feature_progress
    )
    test_features = build_forecast_features(
        history=train_all,
        future_rows=test_raw,
        metadata=metadata,
        first_date=full_dates.min(),
        forecast_origin=full_dates.max(),
    )
    full_train_features.to_csv(DATA_DIR / "orders_train_features.csv", index=False)
    test_features.to_csv(DATA_DIR / "orders_test_features.csv", index=False)

    full_train_features = full_train_features.loc[full_train_features.IsOpen.eq(1)].copy()
    full_target = pd.to_numeric(full_train_features.OrderVolume, errors="raise")
    full_X, test_X, _ = prepare_features(full_train_features, test_features)
    full_xgb = xgb.XGBRegressor(
        objective="reg:squarederror", n_estimators=N_ESTIMATORS,
        learning_rate=0.05, max_depth=8, min_child_weight=100,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
        tree_method="hist", enable_categorical=True, n_jobs=-1,
        random_state=RANDOM_STATE, verbosity=0,
    )
    full_xgb.fit(full_X, full_target.map(math.log1p))
    test_predictions = np.zeros(len(test_features), dtype=float)
    is_open = test_features.IsOpen.eq(1).to_numpy()
    test_predictions[is_open] = np.maximum(
        0, np.expm1(full_xgb.predict(test_X.loc[is_open]))
    )
    submission = sample_submission[["Id"]].merge(
        pd.DataFrame({"Id": test_features.Id, "OrderVolume": test_predictions}),
        on="Id", how="left", validate="one_to_one", sort=False,
    )
    assert submission.OrderVolume.notna().all()
    assert np.isfinite(submission.OrderVolume).all()
    assert (submission.OrderVolume >= 0).all()
    submission.to_csv(DATA_DIR / "submission.csv", index=False)
    full_xgb.save_model(str(DATA_DIR / "xgboost_order_volume_full.json"))
    print("Wrote", len(submission), "submission rows to", DATA_DIR / "submission.csv")
else:
    print("Skipped final submission training. Set RUN_FINAL_SUBMISSION = True when ready.")